In [3]:
from __future__ import annotations
import heapq, math, time
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from IPython.display import display
import pandas as pd

State = Tuple[str, ...]
GOAL: State = ('1','2','3','4','5','6','7','8','b')
W = 3

def neighbors(s: State):
    i=s.index('b'); r,c=divmod(i,W); S=[]
    def sw(a,b): t=list(s); t[a],t[b]=t[b],t[a]; return tuple(t)
    if r>0:S.append(('U',sw(i,i-W)))
    if r<W-1:S.append(('D',sw(i,i+W)))
    if c>0:S.append(('L',sw(i,i-1)))
    if c<W-1:S.append(('R',sw(i,i+1)))
    return S

def is_solvable(s: State):
    a=[x for x in s if x!='b']; inv=0
    for i in range(len(a)):
        for j in range(i+1,len(a)):
            if int(a[i])>int(a[j]): inv+=1
    return inv%2==0

def h_misplaced(s): return sum(1 for i,v in enumerate(s) if v!='b' and v!=GOAL[i])

def h_manhattan(s):
    d=0
    for i,v in enumerate(s):
        if v=='b':continue
        gi=GOAL.index(v); r,c=divmod(i,W); rg,cg=divmod(gi,W)
        d+=abs(r-rg)+abs(c-cg)
    return d

def h_linear_conflict(s):
    m=h_manhattan(s); k=0
    for r in range(W):
        row=s[r*W:(r+1)*W]; L=[]
        for c in range(W):
            v=row[c]
            if v=='b':continue
            gi=GOAL.index(v); gr,gc=divmod(gi,W)
            if gr==r:L.append((gc,c))
        for i in range(len(L)):
            for j in range(i+1,len(L)):
                if L[i][0]>L[j][0] and L[i][1]<L[j][1]: k+=1
    for c in range(W):
        col=s[c::W]; L=[]
        for r in range(W):
            v=col[r]
            if v=='b':continue
            gi=GOAL.index(v); gr,gc=divmod(gi,W)
            if gc==c:L.append((gr,r))
        for i in range(len(L)):
            for j in range(i+1,len(L)):
                if L[i][0]>L[j][0] and L[i][1]<L[j][1]: k+=1
    return m+2*k

HEURISTICS={
    "Heuristic 1 (Misplaced Tiles)": h_misplaced,
    "Heuristic 2 (Manhattan Distance)": h_manhattan,
    "Heuristic 3 (Linear Conflict + Manhattan)": h_linear_conflict,
}

@dataclass(order=True)
class Node:
    f:int; s:State=field(compare=False); g:int=field(compare=False,default=0)

def reconstruct(par:Dict[State,Optional[State]], s:State):
    p=[s]
    while par[s] is not None: s=par[s]; p.append(s)
    p.reverse(); return p

def best_first(start:State,h,max_exp=50000):
    if not is_solvable(start): return None,0,0
    pq=[]; heapq.heappush(pq,Node(h(start),start,0))
    par={start:None}; seen=set(); exp=0
    g_score={start:0}
    while pq and exp<max_exp:
        n=heapq.heappop(pq); s=n.s
        if s in seen: continue
        seen.add(s); exp+=1
        if s==GOAL: return reconstruct(par,s),exp,len(seen)
        for _,ns in neighbors(s):
            new_g = g_score[s] + 1
            if ns in seen: continue
            if ns not in g_score or new_g < g_score[ns]:
                g_score[ns] = new_g
                par[ns]=s
                heapq.heappush(pq,Node(h(ns),ns,new_g))
    return None,exp,len(seen)

def a_star(start:State,h,max_exp=50000):
    if not is_solvable(start): return None,0,0
    pq=[]; g={start:0}; par={start:None}
    heapq.heappush(pq,Node(h(start),start,0))
    closed=set(); exp=0
    while pq and exp<max_exp:
        n=heapq.heappop(pq); s=n.s
        if s in closed: continue
        closed.add(s); exp+=1
        if s==GOAL: return reconstruct(par,s),exp,len(closed)
        gs=g[s]
        for _,ns in neighbors(s):
            ng=gs+1
            if ns in closed and ng>=g.get(ns,10**9): continue
            if ng<g.get(ns,10**9):
                g[ns]=ng; par[ns]=s
                heapq.heappush(pq,Node(ng+h(ns),ns,ng))
    return None,exp,len(closed)

def path_table(path:List[State]):
    rows=[]
    for step,st in enumerate(path):
        r={"Step":step}
        for i,v in enumerate(st,1): r[f"Pos{i}"]=("" if v=="b" else v)
        rows.append(r)
    return pd.DataFrame(rows)

def run_section(title,algo,initials):
    print(f"\n===== {title} =====")
    for hname,hfunc in HEURISTICS.items():
        print(f"\n-- {hname} --")
        lengths=[]
        for i,s in enumerate(initials,1):
            path,exp,_=algo(s,hfunc,80000)
            if path is None:
                print(f"[Initial {i}] No solution found (exp={exp})")
                continue
            steps=len(path)-1
            lengths.append(steps)

            # Formatted sequence matching prompt guidelines
            path_str = " → ".join(["(" + " ".join(state) + ")" for state in path])
            print(f"[Initial {i}] Solution Path:\n{path_str}")
            print(f"Steps taken: {steps}, Nodes expanded: {exp}\n")

            df=path_table(path)
            display(df)
        avg=sum(lengths)/len(lengths) if lengths else 0.0
        print(f"Average number of steps: {avg:.2f}\n")

def main():
    INITIAL_STATES=[
        ('4','5','b','6','1','8','7','3','2'),
        ('1','2','3','4','5','6','b','7','8'),
        ('1','2','3','5','b','6','4','7','8'),
        ('1','3','6','5','2','b','4','7','8'),
        ('7','2','4','5','b','6','8','3','1'),
        ('2','8','1','b','4','3','7','6','5'),
    ]
    t0=time.time()
    run_section("Best-First Search", best_first, INITIAL_STATES)
    run_section("A* Search", a_star, INITIAL_STATES)
    print(f"Total runtime: {time.time()-t0:.2f}s")

if __name__ == '__main__':
    main()


===== Best-First Search =====

-- Heuristic 1 (Misplaced Tiles) --
[Initial 1] No solution found (exp=0)
[Initial 2] Solution Path:
(1 2 3 4 5 6 b 7 8) → (1 2 3 4 5 6 7 b 8) → (1 2 3 4 5 6 7 8 b)
Steps taken: 2, Nodes expanded: 3



,Step,Pos1,Pos2,Pos3,Pos4,Pos5,Pos6,Pos7,Pos8,Pos9
0,0,1,2,3,4,5,6,,7,8
1,1,1,2,3,4,5,6,7,,8
2,2,1,2,3,4,5,6,7,8,


[Initial 3] Solution Path:
(1 2 3 5 b 6 4 7 8) → (1 2 3 b 5 6 4 7 8) → (1 2 3 4 5 6 b 7 8) → (1 2 3 4 5 6 7 b 8) → (1 2 3 4 5 6 7 8 b)
Steps taken: 4, Nodes expanded: 5



,Step,Pos1,Pos2,Pos3,Pos4,Pos5,Pos6,Pos7,Pos8,Pos9
0,0,1,2,3,5,,6,4,7,8
1,1,1,2,3,,5,6,4,7,8
2,2,1,2,3,4,5,6,,7,8
3,3,1,2,3,4,5,6,7,,8
4,4,1,2,3,4,5,6,7,8,


[Initial 4] Solution Path:
(1 3 6 5 2 b 4 7 8) → (1 3 b 5 2 6 4 7 8) → (1 b 3 5 2 6 4 7 8) → (1 2 3 5 b 6 4 7 8) → (1 2 3 b 5 6 4 7 8) → (1 2 3 4 5 6 b 7 8) → (1 2 3 4 5 6 7 b 8) → (1 2 3 4 5 6 7 8 b)
Steps taken: 7, Nodes expanded: 8



...

[Initial 4] Solution Path:
(1 3 6 5 2 b 4 7 8) → (1 3 b 5 2 6 4 7 8) → (1 b 3 5 2 6 4 7 8) → (1 2 3 5 b 6 4 7 8) → (1 2 3 b 5 6 4 7 8) → (1 2 3 4 5 6 b 7 8) → (1 2 3 4 5 6 7 b 8) → (1 2 3 4 5 6 7 8 b)
Steps taken: 7, Nodes expanded: 8



,Step,Pos1,Pos2,Pos3,Pos4,Pos5,Pos6,Pos7,Pos8,Pos9
0,0,1,3,6,5,2,,4,7,8
1,1,1,3,,5,2,6,4,7,8
2,2,1,,3,5,2,6,4,7,8
3,3,1,2,3,5,,6,4,7,8
4,4,1,2,3,,5,6,4,7,8
5,5,1,2,3,4,5,6,,7,8
6,6,1,2,3,4,5,6,7,,8
7,7,1,2,3,4,5,6,7,8,


[Initial 5] Solution Path:
(7 2 4 5 b 6 8 3 1) → (7 2 4 5 3 6 8 b 1) → (7 2 4 5 3 6 8 1 b) → (7 2 4 5 3 b 8 1 6) → (7 2 4 5 b 3 8 1 6) → (7 2 4 b 5 3 8 1 6) → (b 2 4 7 5 3 8 1 6) → (2 b 4 7 5 3 8 1 6) → (2 4 b 7 5 3 8 1 6) → (2 4 3 7 5 b 8 1 6) → (2 4 3 7 b 5 8 1 6) → (2 4 3 7 1 5 8 b 6) → (2 4 3 7 1 5 b 8 6) → (2 4 3 b 1 5 7 8 6) → (2 4 3 1 b 5 7 8 6) → (2 b 3 1 4 5 7 8 6) → (b 2 3 1 4 5 7 8 6) → (1 2 3 b 4 5 7 8 6) → (1 2 3 4 b 5 7 8 6) → (1 2 3 4 5 b 7 8 6) → (1 2 3 4 5 6 7 8 b)
Steps taken: 20, Nodes expanded: 154



,Step,Pos1,Pos2,Pos3,Pos4,Pos5,Pos6,Pos7,Pos8,Pos9
0,0,7,2,4,5,,6,8,3,1
1,1,7,2,4,5,3,6,8,,1
2,2,7,2,4,5,3,6,8,1,
3,3,7,2,4,5,3,,8,1,6
4,4,7,2,4,5,,3,8,1,6
5,5,7,2,4,,5,3,8,1,6
6,6,,2,4,7,5,3,8,1,6
7,7,2,,4,7,5,3,8,1,6
8,8,2,4,,7,5,3,8,1,6
9,9,2,4,3,7,5,,8,1,6


[Initial 6] No solution found (exp=0)
Average number of steps: 8.25

Total runtime: 0.71s
